<a href="https://colab.research.google.com/github/timothy-shiwprasad/COMP6905_assigment_1/blob/main/Assigment_1_Question_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Question 2 : Locating a Pawi Nest





## Question 2A

## Define our decision variables
- x_n : X posiiton of Pawi Nest
- y_n:  Y position of Pawi Nest
- A : Expected interaction intensity
- $\lambda$ : intensity decay (dictates how quickly the interaction intensity decreases with distance)

### Let:
- Q  = (x_n,y_n,A,$\lambda$)
- L = $\lambda$
- $I^i$ : Our Observed Intensity
- $I_i$: Function output


## Function to Calculate Expected Intensity

$I^i$ = A* exp(-L*d_i)

$d_i = \sqrt{(x_i - x_N)^2 + (y_i - y_N)^2}$

## Constraints:
- A > 0
- L > 0

## Calculating the Residuals and Error Rate
Since we are trying to minimize the error rate we will first calculate the residuals.The residuals are the differece between the Expected Intensity function,$I_i$,  using parameters provided and the actual results from our table. We will sum the total of all the differnces squared and this will be our Error Rate.

### Residual Function:
$I_i$: Function output.

$r_i(x_N, y_N, A, \lambda) = I^i - I_i =  A* exp(λ*d_i) - I_i$

### Error Function
$F(x_N, y_N, A, \lambda) = 0.5 *\sum_{i=1}^{25} (r_i(x_N, y_N, A, \lambda))^2$



## Finally Objective Function
Our Goal is to find the parameters that will give the smallest error rate. Therefore our objective function can be defiend as:

$\min_{x_n, y_n, A, \lambda} F(x_N, y_N, A, \lambda)$

Subject to:

- A > 0
- $\lambda$ > 0

## Question 2 B
### Re-parameterize

### New Parameters
$\alpha \in \mathbb{R}, \beta \in \mathbb{R}$

We defined them as $\alpha$ = A and $\beta$ = $\lambda$
### New Model:

$I^i$(θ) = $exp^\alpha * exp(-exp^\beta *d_i)$

### New parameters
θ = (x_n,y_n,α,β)


## Justification of modifications
The changes of constraints to parameters is justified as the $exp^\alpha$ and $exp^β$ gurantees that the values will remain positive and the model is able to serch with no constraints.

In [1]:
# Using Newton's Method to solve a multivariate unconstrained optimization
# problem with JAX.
#
# There are two important adjustments to the standard Newton method:
# 1. Check whether the Hessian produces a valid downhill search direction.
#    If necessary, modify the Hessian so that it is positive definite.
# 2. Use a backtracking line search to determine how far to move in the
#    chosen search direction.


import jax
import jax.numpy as jnp

# Enable 64-bit floating point calculations.
# This gives us greater numerical precision, which can be useful when
# calculating gradients, Hessians, and small changes near a minimum.
jax.config.update("jax_enable_x64", True)

print("JAX version:", jax.__version__)


# ============================================================
# Create Key Variables
# ============================================================

# Measured intensity values from the sensors.
# Each entry corresponds to a sensor located at a particular (x, y)
# coordinate.
observed_intensities = jnp.array([
    [12.9, 21.8, 24.5, 17.2, 19.6],
    [26.0, 32.0, 39.7, 34.1, 27.9],
    [26.8, 44.2, 64.1, 61.7, 33.6],
    [28.5, 44.6, 76.5, 62.7, 36.9],
    [24.4, 32.3, 44.1, 42.0, 30.8],
], dtype=jnp.float64)


# The sensors are positioned at x and y coordinates:
# [0, 2, 4, 6, 8].
coordinates = jnp.arange(0, 9, 2)

# meshgrid creates a 5 x 5 grid of all possible (x, y) sensor locations.
X1, Y1 = jnp.meshgrid(coordinates, coordinates)

# Flatten the coordinate grids so that each sensor has one x-coordinate
# and one y-coordinate.
sensor_x = X1.ravel()
sensor_y = Y1.ravel()

# Flatten the observed intensity matrix so that the observations line up
# with the flattened sensor coordinates.
observations = observed_intensities.ravel()


# ============================================================
# Objective Function
# ============================================================

def objective(theta):
    r"""
    Calculate the objective function (Half Sum of Squared Errors).

    theta contains four unknown parameters:
        x_n    = x-coordinate of the source
        y_n    = y-coordinate of the source
        alpha  = log(A)
        beta   = log(lambda)

    A and lambda are recovered using exponentials:
        A      = exp(alpha)
        lambda = exp(beta)

    Using alpha and beta instead of A and lambda means the optimization
    is unconstrained. No explicit constraint such as A > 0 or lambda > 0
    is required because exponentials are always positive.
    """

    # Extract the four optimization variables.
    x_n, y_n, alpha, beta = theta

    # Transform alpha and beta back into the physical parameters A
    # and lambda. This guarantees that A > 0 and lambda > 0.
    A = jnp.exp(alpha)
    lam = jnp.exp(beta)

    # Difference between each sensor location and the estimated
    # source location.
    dx = sensor_x - x_n
    dy = sensor_y - y_n

    # Euclidean distance from the estimated source to each sensor.
    #
    # The very small 1e-24 term prevents the square root from receiving
    # exactly zero in situations where the source is located directly
    # on a sensor.
    distance = jnp.sqrt(dx**2 + dy**2 + 1e-24)

    # Calculate the predicted intensity at each sensor.
    #
    # The model assumes intensity decreases exponentially with distance.
    predicted = A * jnp.exp(-lam * distance)

    # Difference between the model predictions and the observed data.
    residual = predicted - observations

    # Return half of the Sum of Squared Errors (SSE).
    #
    # The factor 0.5 does not change the location of the minimum.
    # It simply makes the derivatives cleaner because the 2 from
    # differentiating the square cancels with the 0.5.
    return 0.5 * jnp.sum(residual**2)


# ============================================================
# Gradient and Hessian
# ============================================================

def gradient_hessian(objective):
    r"""
    Create the gradient and Hessian functions using JAX.

    The gradient contains the first derivative of the objective with
    respect to each optimization variable.

    The Hessian contains all second-order partial derivatives.
    Newton's method uses both the gradient and Hessian to determine
    the next search direction.
    """

    # JAX automatically differentiates the objective function.
    _gradient = jax.grad(objective)

    # JAX automatically calculates the matrix of second derivatives.
    _hessian = jax.hessian(objective)

    return _gradient, _hessian


# ============================================================
# Newton's Method
# ============================================================

def newton_method(initial_theta, tol=1e-8, max_iteration=100):

    theta = jnp.array(initial_theta, dtype=jnp.float64)

    # Identity matrix used when modifying the Hessian.
    # Adding shift * I to the Hessian increases its eigenvalues and
    # can make it positive definite.
    identity = jnp.eye(4, dtype=jnp.float64)

    # Create the gradient and Hessian functions.
    _gradient, _hessian = gradient_hessian(objective)

    for i in range(max_iteration):

        # Evaluate the objective function at the current point.
        value = objective(theta)

        # Calculate the gradient at the current point.
        gradient = _gradient(theta)


        # ----------------------------------------------------
        # Check for convergence
        # ---------------------------------------------------
        # At a minimum, the gradient should be approximately zero.
        #
        # We use a tolerance instead of requiring exactly zero because
        # floating-point calculations rarely produce an exact zero.
        # max(abs(gradient)) checks the largest absolute gradient
        # component.
        if float(jnp.max(jnp.abs(gradient))) < tol:
            return theta, value, i, True


        # ----------------------------------------------------
        # Calculate and adjust the Hessian
        # ----------------------------------------------------

        # Calculate the Hessian matrix at the current point.
        hessian = _hessian(theta)

        # The eigenvalues of the Hessian tell us about its curvature.
        #
        # If the smallest eigenvalue is negative, the Hessian is not
        # positive definite. In that case, the Newton direction may
        # fail to be a downhill direction.
        smallest_eigenvalue = jnp.min(jnp.linalg.eigvalsh(hessian))

        print(smallest_eigenvalue)


        # If the smallest eigenvalue is too small, add a positive
        # multiple of the identity matrix to shift all eigenvalues
        # upward.
        #
        # This gives us a better-conditioned Hessian and helps ensure
        # that the resulting Newton direction is a downhill direction.
        shift = jnp.maximum(0.0, 1e-3 - smallest_eigenvalue)

        adjusted_hessian = hessian + shift * identity


        # ----------------------------------------------------
        # Calculate the Newton Search Direction
        # ----------------------------------------------------

        # Standard Newton's method solves:
        #
        #       H p = -gradient
        #
        # where:H = Hessian , p = search direction

        direction = jnp.linalg.solve(
            adjusted_hessian,
            -gradient
        )


        # Check whether the direction is actually downhill.
        # A direction is downhill when:
        # gradient · direction < 0
        # If it is >= 0, moving in this direction would not decrease
        # the objective function.
        directional_derivative = jnp.dot(gradient, direction)
        # Stop if:
        #The calculated direction contains NaN or infinity
        #The direction is not a downhill direction.
        if (
            not bool(jnp.all(jnp.isfinite(direction)))
            or float(directional_derivative) >= 0
        ):
            return theta, value, i, False


        # ----------------------------------------------------
        # Backtracking Line Search
        # ----------------------------------------------------

        # Start by trying the full Newton step.
        step_size = 1.0
        accepted = False


        # If the full step does not sufficiently decrease the
        # objective function, repeatedly reduce the step size.
        while step_size > 1e-12:

            # Proposed new point.
            candidate = theta + step_size * direction

            # Evaluate the objective at the proposed point.
            candidate_value = objective(candidate)


            # Armijo-style sufficient decrease condition.
            #
            # We accept the step if:
            # 1. The candidate objective is finite.
            # 2. The objective decreases by a sufficient amount.
            #
            # The 1e-4 factor controls how much decrease we require
            # before accepting the step.
            if (
                bool(jnp.isfinite(candidate_value))
                and float(candidate_value)
                <= float(
                    value
                    + 1e-4 * step_size * directional_derivative
                )
            ):
                accepted = True
                break

            # The step was too large, so cut it in half and try again.
            step_size *= 0.5


        # If no acceptable step was found, stop the algorithm.
        if not accepted:
            return theta, value, i, False


        # Move to the accepted candidate point.
        theta = candidate


    # Maximum number of iterations reached without satisfying
    # the convergence tolerance.
    return theta, objective(theta), i, False


# ============================================================
# Try Multiple Initial Guesses
# ============================================================

# Newton's method can behave differently depending on the starting
# point. We therefore test several initial guesses.
#
# Each tuple contains:
#     x0     = initial x-coordinate
#     y0     = initial y-coordinate
#     A0     = initial amplitude
#     lam0   = initial lambda
starts = [
    (4.0, 5.0, 80.0, 0.30),
    (1.0, 1.0, 40.0, 0.15),
    (4.5, 5.5, 75.0, 0.20),
]


# Run Newton's method from each starting point.
for x0, y0, A0, lam0 in starts:
    initial_theta = jnp.array([
        x0,
        y0,
        jnp.log(A0),
        jnp.log(lam0)
    ])
    theta, loss, iterations, converged = newton_method(initial_theta)


    # Extract the optimized parameters.
    x_n, y_n, alpha, beta = theta

    # Convert alpha and beta back into A and lambda.
    A = jnp.exp(alpha)
    lam = jnp.exp(beta)


    # --------------------------------------------------------
    # Display Results
    # --------------------------------------------------------

    print(f"Start: x={x0}, y={y0}, A={A0}, lambda={lam0}")

    print(
        f"  x_N={float(x_n):.6f}, "
        f"y_N={float(y_n):.6f}"
    )

    print(
        f"  A={float(A):.6f}, "
        f"lambda={float(lam):.6f}"
    )

    print(f"  Half-SSE={float(loss):.6f}")

    print(
        f"  Iterations={iterations}, "
        f"converged={converged}\n"
    )

JAX version: 0.11.1
1065.588834245297
768.9206036423518
1152.7370792026566
1329.7163483281274
1307.7598059004213
1307.5866136251043
Start: x=4.0, y=5.0, A=80.0, lambda=0.3
  x_N=4.594074, y_N=5.268285
  A=96.341489, lambda=0.269724
  Half-SSE=58.458694
  Iterations=6, converged=True

-159.52495851964665
-5108.65260372939
224.78744856640662
1937.5577261101746
1511.5314652866346
1372.4636422863675
1310.2831622717356
1307.590242604301
Start: x=1.0, y=1.0, A=40.0, lambda=0.15
  x_N=4.594074, y_N=5.268285
  A=96.341489, lambda=0.269724
  Half-SSE=58.458694
  Iterations=8, converged=True

647.3653475619504
1526.6628395230268
1356.9358378065763
1308.7653228933352
1307.5873373024222
Start: x=4.5, y=5.5, A=75.0, lambda=0.2
  x_N=4.594074, y_N=5.268285
  A=96.341489, lambda=0.269724
  Half-SSE=58.458694
  Iterations=5, converged=True



## Question 2 C

### Observations

The same solution is obtained for the 3 starting points used. The final x and y postions are quite close . They are 4.6 , 5.2 when rounded and the expected interaction intensity is 96 when rounded and the decay is 0.3 when rounded. The values are very close and we can assumen the nest can be found near the point [4.6,5.2]